# TP : Transfer Learning sur FashionMNIST
## AlexNet, ResNet18, VGG16 — Gel total, Gel partiel, Fine-tuning progressif

| Modèle | Année | Paramètres | Particularité |
|--------|-------|-----------|---------------|
| AlexNet | 2012 | ~60M | Premier grand succès du Deep Learning |
| ResNet18 | 2015 | ~11M | Skip connections (connexions résiduelles) |
| VGG16 | 2014 | ~138M | Très profond, excellent extracteur de features |

**Problème à résoudre :**
- FashionMNIST = images 28×28 grises (1 canal)
- Modèles ImageNet = images 224×224 RGB (3 canaux), normalisées, 1000 classes
- → Il faut adapter les données ET la dernière couche du modèle


## 1. Setup et imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
import copy
import time

%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 2. Préparation des données pour modèles pré-entraînés


In [ ]:
# Transformations: 28x28 gray -> 224x224 RGB + ImageNet normalization
data_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=data_transform)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=data_transform)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Verify
img, lbl = train_dataset[0]
print(f"Image shape: {img.shape}")   # [3, 224, 224]
print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")


## 3. Fonctions utilitaires (entraînement, évaluation, résumé)


In [ ]:
def count_params(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total

def run_experiment(model, name, n_epochs=5, lr=0.001):
    """Full training loop. Returns history dict."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    # Only optimize unfrozen parameters
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    total_p, train_p = count_params(model)
    print(f"\n{'='*65}")
    print(f"  {name}")
    print(f"  Params: {train_p:,} trainable / {total_p:,} total")
    print(f"{'='*65}")

    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    start = time.time()

    for epoch in range(n_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['test_loss'].append(te_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
        print(f"  Epoch {epoch+1}/{n_epochs} | "
              f"Train: {tr_acc:.1f}% loss={tr_loss:.4f} | "
              f"Test: {te_acc:.1f}% loss={te_loss:.4f}")

    elapsed = time.time() - start
    print(f"  Time: {elapsed:.0f}s | Best test acc: {max(history['test_acc']):.1f}%")
    return history

def get_all_predictions(model, loader, device):
    """Get predictions for entire dataset."""
    model.to(device)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.append(preds.cpu())
            all_labels.append(labels)
    return torch.cat(all_preds), torch.cat(all_labels)

def show_confusion_matrix(model, loader, device, title):
    """Print classification report + plot confusion matrix."""
    preds, labels = get_all_predictions(model, loader, device)

    print(classification_report(labels, preds, target_names=classes))

    cm = confusion_matrix(labels.numpy(), preds.numpy())
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', square=True,
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.show()


---
## 4. Expérience 1 : AlexNet — Tout gelé sauf la dernière couche

**Stratégie :** On gèle toutes les couches du modèle pré-entraîné. On remplace uniquement la dernière couche `classifier[6]` (Linear 4096→1000) par une nouvelle couche (Linear 4096→10). Seule cette nouvelle couche sera entraînée.


In [ ]:
# Load pretrained AlexNet
alexnet_frozen = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# FREEZE all layers
for param in alexnet_frozen.parameters():
    param.requires_grad = False

# REPLACE last layer (new layer has requires_grad=True by default)
alexnet_frozen.classifier[6] = nn.Linear(4096, 10)

# Train
hist_alexnet_frozen = run_experiment(
    alexnet_frozen,
    "AlexNet — Tout gelé sauf classifier[6]",
    n_epochs=5, lr=0.001
)


---
## 5. Expérience 2 : ResNet18 — Tout gelé sauf fc

**Stratégie :** Geler tout le réseau. Remplacer `model.fc` (Linear 512→1000) par (Linear 512→10).


In [ ]:
resnet_frozen = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in resnet_frozen.parameters():
    param.requires_grad = False

resnet_frozen.fc = nn.Linear(resnet_frozen.fc.in_features, 10)

hist_resnet_frozen = run_experiment(
    resnet_frozen,
    "ResNet18 — Tout gelé sauf fc",
    n_epochs=5, lr=0.001
)


---
## 6. Expérience 3 : VGG16 — Tout gelé sauf la dernière couche

**Stratégie :** Geler tout. Remplacer `classifier[6]` (Linear 4096→1000) par (Linear 4096→10).


In [ ]:
vgg_frozen = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

for param in vgg_frozen.parameters():
    param.requires_grad = False

vgg_frozen.classifier[6] = nn.Linear(4096, 10)

hist_vgg_frozen = run_experiment(
    vgg_frozen,
    "VGG16 — Tout gelé sauf classifier[6]",
    n_epochs=5, lr=0.001
)


---
## 7. Gel partiel — Geler seulement les premières couches

**Stratégie :** On gèle les couches basses (qui détectent des features universelles : bords, textures) et on dégèle les couches hautes + la dernière couche pour qu'elles s'adaptent à notre tâche.

| Modèle | Couches gelées | Couches dégelées |
|--------|---------------|-----------------|
| AlexNet | `features` | `classifier` entier |
| ResNet18 | conv1 → layer3 | `layer4` + `fc` |
| VGG16 | `features[:24]` (premiers blocs conv) | `features[24:]` + `classifier` |


In [ ]:
# --- AlexNet gel partiel : features gelées, tout le classifier libre ---
alexnet_partial = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Freeze features only
for param in alexnet_partial.features.parameters():
    param.requires_grad = False
# Classifier stays unfrozen (requires_grad=True by default)

# Replace last layer
alexnet_partial.classifier[6] = nn.Linear(4096, 10)

hist_alexnet_partial = run_experiment(
    alexnet_partial,
    "AlexNet — Gel partiel (features gelées, classifier libre)",
    n_epochs=5, lr=0.001
)


In [ ]:
# --- ResNet18 gel partiel : layer4 + fc dégelés ---
resnet_partial = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all
for param in resnet_partial.parameters():
    param.requires_grad = False

# Unfreeze layer4 + fc
for param in resnet_partial.layer4.parameters():
    param.requires_grad = True

resnet_partial.fc = nn.Linear(resnet_partial.fc.in_features, 10)

hist_resnet_partial = run_experiment(
    resnet_partial,
    "ResNet18 — Gel partiel (layer4 + fc dégelés)",
    n_epochs=5, lr=0.001
)


In [ ]:
# --- VGG16 gel partiel : premiers blocs conv gelés, derniers conv + classifier libres ---
vgg_partial = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Freeze all
for param in vgg_partial.parameters():
    param.requires_grad = False

# Unfreeze last conv block (features[24:]) + classifier
for param in vgg_partial.features[24:].parameters():
    param.requires_grad = True
for param in vgg_partial.classifier.parameters():
    param.requires_grad = True

vgg_partial.classifier[6] = nn.Linear(4096, 10)

hist_vgg_partial = run_experiment(
    vgg_partial,
    "VGG16 — Gel partiel (dernier bloc conv + classifier dégelés)",
    n_epochs=5, lr=0.001
)


---
## 8. Fine-tuning progressif

**Stratégie :** On commence avec tout gelé sauf la dernière couche (2 epochs). Puis on dégèle progressivement les couches plus profondes et on continue l'entraînement avec un learning rate plus faible.

C'est la technique qui donne généralement les meilleurs résultats.


In [ ]:
def progressive_finetune(model, name, unfreeze_schedule, n_epochs_per_phase=2):
    """
    Progressive fine-tuning.
    unfreeze_schedule: list of (phase_name, layers_to_unfreeze, lr)
    layers_to_unfreeze: list of model sub-modules to set requires_grad=True
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    total_p, _ = count_params(model)
    print(f"\n{'='*65}")
    print(f"  PROGRESSIVE FINE-TUNING: {name}")
    print(f"  Total params: {total_p:,}")
    print(f"{'='*65}")

    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    start = time.time()
    global_epoch = 0

    for phase_name, layers, lr in unfreeze_schedule:
        # Unfreeze specified layers
        for layer in layers:
            for param in layer.parameters():
                param.requires_grad = True

        _, train_p = count_params(model)
        print(f"\n  --- Phase: {phase_name} | LR={lr} | Trainable: {train_p:,} ---")

        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

        for epoch in range(n_epochs_per_phase):
            global_epoch += 1
            tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            te_loss, te_acc = evaluate(model, test_loader, criterion, device)
            history['train_loss'].append(tr_loss)
            history['test_loss'].append(te_loss)
            history['train_acc'].append(tr_acc)
            history['test_acc'].append(te_acc)
            print(f"    Epoch {global_epoch} | "
                  f"Train: {tr_acc:.1f}% | Test: {te_acc:.1f}%")

    elapsed = time.time() - start
    print(f"\n  Total time: {elapsed:.0f}s | Best test acc: {max(history['test_acc']):.1f}%")
    return history, model


In [ ]:
# --- AlexNet progressive fine-tuning ---
alexnet_prog = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Start fully frozen
for param in alexnet_prog.parameters():
    param.requires_grad = False
alexnet_prog.classifier[6] = nn.Linear(4096, 10)

schedule_alexnet = [
    ("Phase 1: classifier[6] only",   [alexnet_prog.classifier[6]], 0.001),
    ("Phase 2: + full classifier",    [alexnet_prog.classifier],    0.0005),
    ("Phase 3: + last conv features", [alexnet_prog.features[8:]],  0.0001),
]

hist_alexnet_prog, alexnet_prog = progressive_finetune(
    alexnet_prog, "AlexNet", schedule_alexnet, n_epochs_per_phase=2
)


In [ ]:
# --- ResNet18 progressive fine-tuning ---
resnet_prog = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in resnet_prog.parameters():
    param.requires_grad = False
resnet_prog.fc = nn.Linear(resnet_prog.fc.in_features, 10)

schedule_resnet = [
    ("Phase 1: fc only",        [resnet_prog.fc],     0.001),
    ("Phase 2: + layer4",       [resnet_prog.layer4], 0.0005),
    ("Phase 3: + layer3",       [resnet_prog.layer3], 0.0001),
]

hist_resnet_prog, resnet_prog = progressive_finetune(
    resnet_prog, "ResNet18", schedule_resnet, n_epochs_per_phase=2
)


In [ ]:
# --- VGG16 progressive fine-tuning ---
vgg_prog = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

for param in vgg_prog.parameters():
    param.requires_grad = False
vgg_prog.classifier[6] = nn.Linear(4096, 10)

schedule_vgg = [
    ("Phase 1: classifier[6] only",   [vgg_prog.classifier[6]],  0.001),
    ("Phase 2: + full classifier",    [vgg_prog.classifier],     0.0005),
    ("Phase 3: + last conv block",    [vgg_prog.features[24:]],  0.0001),
]

hist_vgg_prog, vgg_prog = progressive_finetune(
    vgg_prog, "VGG16", schedule_vgg, n_epochs_per_phase=2
)


---
## 9. Comparaison de TOUS les résultats

### 9 expériences au total :
- 3 modèles × 3 stratégies (tout gelé, gel partiel, fine-tuning progressif)


In [ ]:
all_experiments = {
    # (history, color, linestyle)
    'AlexNet tout gelé':        (hist_alexnet_frozen,  '#e74c3c', '-'),
    'AlexNet gel partiel':      (hist_alexnet_partial, '#e74c3c', '--'),
    'AlexNet progressif':       (hist_alexnet_prog,    '#e74c3c', ':'),
    'ResNet18 tout gelé':       (hist_resnet_frozen,   '#2ecc71', '-'),
    'ResNet18 gel partiel':     (hist_resnet_partial,  '#2ecc71', '--'),
    'ResNet18 progressif':      (hist_resnet_prog,     '#2ecc71', ':'),
    'VGG16 tout gelé':          (hist_vgg_frozen,      '#3498db', '-'),
    'VGG16 gel partiel':        (hist_vgg_partial,     '#3498db', '--'),
    'VGG16 progressif':         (hist_vgg_prog,        '#3498db', ':'),
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for name, (hist, color, ls) in all_experiments.items():
    axes[0].plot(hist['test_acc'], label=name, color=color, linestyle=ls, linewidth=2)
    axes[1].plot(hist['test_loss'], label=name, color=color, linestyle=ls, linewidth=2)

axes[0].set_title('Test Accuracy par Epoch', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Test Loss par Epoch', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparaison : 3 modèles × 3 stratégies de gel', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Summary table
print(f"{'Modèle':<35} {'Stratégie':<20} {'Best Test Acc':>13} {'Final Test Acc':>15}")
print("=" * 85)

rows = [
    ('AlexNet',  'Tout gelé',    hist_alexnet_frozen),
    ('AlexNet',  'Gel partiel',  hist_alexnet_partial),
    ('AlexNet',  'Progressif',   hist_alexnet_prog),
    ('ResNet18', 'Tout gelé',    hist_resnet_frozen),
    ('ResNet18', 'Gel partiel',  hist_resnet_partial),
    ('ResNet18', 'Progressif',   hist_resnet_prog),
    ('VGG16',    'Tout gelé',    hist_vgg_frozen),
    ('VGG16',    'Gel partiel',  hist_vgg_partial),
    ('VGG16',    'Progressif',   hist_vgg_prog),
]

for model_name, strategy, hist in rows:
    best = max(hist['test_acc'])
    final = hist['test_acc'][-1]
    print(f"{model_name:<35} {strategy:<20} {best:>12.1f}% {final:>14.1f}%")


### Tableau récapitulatif des modèles

| Modèle | Année | Paramètres totaux | Particularité | Dernière couche à modifier |
|--------|-------|------------------|---------------|---------------------------|
| AlexNet | 2012 | ~60M | Premier grand succès Deep Learning | `classifier[6]` |
| ResNet18 | 2015 | ~11M | Skip connections | `fc` |
| VGG16 | 2014 | ~138M | Excellent extracteur de features | `classifier[6]` |

### Résumé des stratégies de gel

| Stratégie | Description | Quand l'utiliser |
|-----------|------------|-----------------|
| **Tout geler** | Seule la dernière couche est entraînée | Peu de données, domaine similaire à ImageNet |
| **Gel partiel** | Couches basses gelées, couches hautes libres | Assez de données, domaine modérément différent |
| **Fine-tuning progressif** | Dégeler progressivement couche par couche | Meilleur résultat possible |


---
## 10. Évaluation finale — Confusion Matrix + Classification Report

On évalue les meilleurs modèles de chaque architecture.


In [ ]:
# Find the best experiment
best_name = max(rows, key=lambda r: max(r[2]['test_acc']))
print(f"Best overall: {best_name[0]} ({best_name[1]}) — {max(best_name[2]['test_acc']):.1f}%")


In [ ]:
# Confusion matrix for AlexNet progressive
print("\n" + "="*50)
print("  AlexNet — Progressive Fine-Tuning")
print("="*50)
show_confusion_matrix(alexnet_prog, test_loader, device,
                      "AlexNet Progressive — Confusion Matrix")


In [ ]:
# Confusion matrix for ResNet18 progressive
print("\n" + "="*50)
print("  ResNet18 — Progressive Fine-Tuning")
print("="*50)
show_confusion_matrix(resnet_prog, test_loader, device,
                      "ResNet18 Progressive — Confusion Matrix")


In [ ]:
# Confusion matrix for VGG16 progressive
print("\n" + "="*50)
print("  VGG16 — Progressive Fine-Tuning")
print("="*50)
show_confusion_matrix(vgg_prog, test_loader, device,
                      "VGG16 Progressive — Confusion Matrix")


---
## 11. Conclusion

Ce TP a démontré :

1. **Le Transfer Learning** permet d'atteindre de hautes performances même sur un dataset très différent d'ImageNet (images grises 28×28 de vêtements).

2. **Le gel des couches** est crucial :
   - Tout geler = rapide mais performance limitée
   - Gel partiel = bon compromis
   - Fine-tuning progressif = meilleurs résultats

3. **L'adaptation des données** est indispensable : resize, conversion gris→RGB, normalisation ImageNet.

4. **La modification de la dernière couche** est obligatoire : 1000 classes ImageNet → 10 classes FashionMNIST.
